# CSE 814 — Machine Learning Lab
## Lab 1 — Python for Machine Learning: NumPy, Pandas & Exploratory Data Analysis

**Department of Computer Science & Engineering, University of Chittagong**
Co-requisite of **CSE 815 — Machine Learning (Theory)** · Instructor: Rokan Uddin Faruqui

---

| | |
|---|---|
| **CLO addressed** | **CLO1** — Implement core machine learning algorithms from scratch using Python |
| **Aligned lectures** | Theory Weeks 1–2 (the learning problem; linear algebra & probability foundations) |
| **Duration** | One 3-hour lab session |
| **Weight** | 2% (Lab Task 1) |
| **Assessed in** | Quiz 1 (Week 7), together with Labs 2–3 |

### Learning objectives

By the end of this lab you should be able to:

1. Run and document work in a Jupyter notebook.
2. Create, index, slice and reshape **NumPy** arrays, and explain why *vectorised* code
   is the foundation of every ML library.
3. Use **broadcasting** and NumPy's linear-algebra routines — the same operations you will
   implement by hand in Lab 2 (linear regression).
4. Load, clean, filter, group and merge tabular data with **Pandas**.
5. Carry out a first **exploratory data analysis (EDA)**: distributions, outliers,
   correlations, and class separability — and say what those tell you *before* you model.

### Deliverable

A completed copy of this notebook, renamed `Lab01_<your-roll-number>.ipynb`, with:

* every **Exercise** cell filled in and executed (outputs visible),
* every **Question** answered in the markdown cell provided,
* the Section 6 mini-EDA written up in your own words.

> **How to work through this notebook.** Read a section, run the worked examples
> (`Shift`+`Enter`), then do the exercises for that section. Do not just run every cell top
> to bottom — change values, break things, and see what happens. That is the point of a lab.


---
## 1. Environment check

This course uses **Python 3**, with Anaconda or a virtual environment. Everything in this lab
runs offline: all datasets are bundled with scikit-learn, so no download is required.

If a package is reported as missing below, install it from a terminal:

```bash
pip install numpy pandas matplotlib seaborn scikit-learn
# or, with Anaconda:
conda install numpy pandas matplotlib seaborn scikit-learn
```


In [ ]:
import sys
import importlib

print("Python", sys.version.split()[0])
print("-" * 46)

for pkg in ["numpy", "pandas", "matplotlib", "seaborn", "sklearn"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<12} {getattr(mod, '__version__', 'unknown'):>10}   OK")
    except ImportError:
        print(f"{pkg:<12} {'--':>10}   MISSING  <-- install this")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Reproducibility: fix the seed so every run of this notebook gives identical numbers.
# In ML this matters far more than most beginners expect -- results you cannot reproduce
# are results you cannot defend.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Readable display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 3)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

print("Setup complete.")

---
## 2. NumPy — the numerical foundation

Every ML library you will meet (scikit-learn, TensorFlow, PyTorch) is built on the idea
NumPy introduced: store numbers in a **contiguous, typed, n-dimensional array** and operate
on the whole array at once.

In machine learning notation, a dataset is a matrix

$$\mathbf{X} \in \mathbb{R}^{n \times d}$$

with **$n$ rows (samples)** and **$d$ columns (features)**, and a target vector
$\mathbf{y} \in \mathbb{R}^{n}$. In NumPy that is exactly `X.shape == (n, d)` and
`y.shape == (n,)`. Keeping those shapes straight is most of the debugging you will do
this semester.

### 2.1 Creating arrays

In [ ]:
# From a Python list
a = np.array([1, 2, 3, 4, 5])
print("a          =", a)
print("shape      =", a.shape)     # (5,)  -- a 1-D array, NOT (5,1) and NOT (1,5)
print("ndim       =", a.ndim)
print("dtype      =", a.dtype)
print("size       =", a.size)

# 2-D array: 3 samples, 4 features
X = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
print("\nX shape    =", X.shape, " -> n =", X.shape[0], "samples,", X.shape[1], "features")

In [4]:
# Common constructors
print("zeros     :", np.zeros(4))
print("ones      :", np.ones((2, 3)), sep="\n")
print("full      :", np.full(3, 7.5))
print("eye (I)   :", np.eye(3), sep="\n")
print("arange    :", np.arange(0, 10, 2))        # start, stop (exclusive), step
print("linspace  :", np.linspace(0, 1, 5))       # start, stop (inclusive), how many

# Random numbers -- always via a seeded Generator, never the legacy np.random.rand
print("\nuniform  :", rng.random(4).round(3))
print("normal    :", rng.normal(loc=0, scale=1, size=4).round(3))
print("integers  :", rng.integers(low=0, high=10, size=4))

zeros     : [0. 0. 0. 0.]
ones      :
[[1. 1. 1.]
 [1. 1. 1.]]
full      : [7.5 7.5 7.5]
eye (I)   :
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
arange    : [0 2 4 6 8]
linspace  : [0.   0.25 0.5  0.75 1.  ]

uniform  : [0.774 0.439 0.859 0.697]
normal    : [-1.951 -1.302  0.128 -0.316]
integers  : [5 1 8 4]


### 2.2 Indexing, slicing and boolean masks

Slicing uses `start:stop:step`, and **`stop` is exclusive**. For 2-D arrays the convention
is `X[row, column]` — which for us reads as `X[sample, feature]`.

In [5]:
v = np.arange(10) * 10
print("v            =", v)
print("v[0]         =", v[0])          # first
print("v[-1]        =", v[-1])         # last
print("v[2:5]       =", v[2:5])        # elements 2,3,4
print("v[:3]        =", v[:3])         # first three
print("v[::2]       =", v[::2])        # every second
print("v[::-1]      =", v[::-1])       # reversed

print("\nX =", X, sep="\n")
print("\nX[0, 0]      =", X[0, 0])          # first sample, first feature
print("X[1]         =", X[1])                # entire second sample (a row)
print("X[:, 2]      =", X[:, 2])             # feature 2 across all samples (a column)
print("X[0:2, 1:3]  =", X[0:2, 1:3], sep="\n")

v            = [ 0 10 20 30 40 50 60 70 80 90]
v[0]         = 0
v[-1]        = 90
v[2:5]       = [20 30 40]
v[:3]        = [ 0 10 20]
v[::2]       = [ 0 20 40 60 80]
v[::-1]      = [90 80 70 60 50 40 30 20 10  0]

X =
[[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]]

X[0, 0]      = 1.0
X[1]         = [5. 6. 7. 8.]
X[:, 2]      = [ 3.  7. 11.]
X[0:2, 1:3]  =
[[2. 3.]
 [6. 7.]]


In [6]:
# Boolean masking -- this is how you filter data without writing a loop
scores = np.array([55, 91, 72, 43, 88, 67, 95, 30])

mask = scores >= 70
print("mask            =", mask)          # an array of True/False
print("passing scores  =", scores[mask])  # keep only where mask is True
print("how many passed =", mask.sum())    # True counts as 1
print("pass rate       =", mask.mean().round(3))

# Conditions combine with & (and), | (or), ~ (not) -- and each side needs brackets
middle = scores[(scores > 50) & (scores < 90)]
print("50 < s < 90     =", middle)

# np.where: vectorised if/else
grade = np.where(scores >= 70, "pass", "fail")
print("grades          =", grade)

mask            = [False  True  True False  True False  True False]
passing scores  = [91 72 88 95]
how many passed = 4
pass rate       = 0.5
50 < s < 90     = [55 72 88 67]
grades          = ['fail' 'pass' 'pass' 'fail' 'pass' 'fail' 'pass' 'fail']


### 2.3 Why vectorise? A timing experiment

This is the single most important habit in this course. The two cells below compute exactly
the same thing — the sum of squares of a million numbers — with a Python loop and with NumPy.

In [7]:
big = rng.random(1_000_000)

def sum_squares_loop(arr):
    total = 0.0
    for x in arr:
        total += x * x
    return total

loop_time = %timeit -o -q -n 1 -r 3 sum_squares_loop(big)
vec_time  = %timeit -o -q -n 1 -r 3 np.sum(big ** 2)

print(f"Python loop : {loop_time.best * 1000:8.2f} ms")
print(f"NumPy       : {vec_time.best * 1000:8.2f} ms")
print(f"Speed-up    : {loop_time.best / vec_time.best:8.1f}x")

# Same answer, very different cost
assert np.isclose(sum_squares_loop(big), np.sum(big ** 2))

Python loop :    44.47 ms
NumPy       :     0.23 ms
Speed-up    :    195.8x


> **Why the difference?** The Python loop interprets bytecode and allocates a boxed object
> per element. NumPy dispatches once into a compiled C loop over a contiguous block of memory.
> When you train a model on 100,000 samples for 1,000 iterations, this factor is the difference
> between *seconds* and *hours*.
>
> **Rule for this course: if you are writing a `for` loop over data points, stop and ask
> whether NumPy can do it in one expression.**

### 2.4 Broadcasting

Broadcasting lets NumPy combine arrays of *different* shapes without copying data. The rule,
compared right-to-left across dimensions: two dimensions are compatible when they are **equal**
or one of them is **1**.

In [8]:
M = np.arange(12).reshape(3, 4).astype(float)
print("M =", M, sep="\n")

print("\nM + 100 (scalar broadcast):", M + 100, sep="\n")

row = np.array([0.0, 10.0, 20.0, 30.0])        # shape (4,)  -> stretched down the 3 rows
print("\nM + row  (shapes (3,4) + (4,)):", M + row, sep="\n")

col = np.array([[1.0], [2.0], [3.0]])          # shape (3,1) -> stretched across the 4 columns
print("\nM + col  (shapes (3,4) + (3,1)):", M + col, sep="\n")

M =
[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]

M + 100 (scalar broadcast):
[[100. 101. 102. 103.]
 [104. 105. 106. 107.]
 [108. 109. 110. 111.]]

M + row  (shapes (3,4) + (4,)):
[[ 0. 11. 22. 33.]
 [ 4. 15. 26. 37.]
 [ 8. 19. 30. 41.]]

M + col  (shapes (3,4) + (3,1)):
[[ 1.  2.  3.  4.]
 [ 6.  7.  8.  9.]
 [11. 12. 13. 14.]]


In [9]:
# The classic ML use of broadcasting: standardising features (z-scores).
#     z = (x - mean) / std,  computed per COLUMN (per feature)
data = rng.normal(loc=[10, 200, 0.5], scale=[2, 50, 0.1], size=(6, 3))

mu    = data.mean(axis=0)   # axis=0 -> collapse the ROWS, giving one value per column
sigma = data.std(axis=0)
z     = (data - mu) / sigma  # (6,3) - (3,) - broadcasting does the rest

print("column means before :", data.mean(axis=0).round(2))
print("column stds  before :", data.std(axis=0).round(2))
print("column means after  :", z.mean(axis=0).round(6))   # ~0
print("column stds  after  :", z.std(axis=0).round(6))    # ~1

column means before : [  9.   229.55   0.51]
column stds  before : [ 2.59 61.65  0.13]
column means after  : [ 0. -0. -0.]
column stds  after  : [1. 1. 1.]


> **`axis` is the dimension you collapse.** For an `(n, d)` matrix, `axis=0` gives you one
> number per *feature* (what you almost always want in ML) and `axis=1` gives one number per
> *sample*. Getting this backwards is the most common bug in this lab.

### 2.5 Linear algebra

These are the operations behind the normal equation you will implement in **Lab 2**:

$$\mathbf{w} = (\mathbf{X}^{\top}\mathbf{X})^{-1}\mathbf{X}^{\top}\mathbf{y}$$

In [10]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([5.0, 10.0])

print("A.T (transpose) =", A.T, sep="\n")
print("\nA @ A  (matrix product) =", A @ A, sep="\n")
print("\nA * A  (ELEMENT-wise!)  =", A * A, sep="\n")   # note the difference
print("\nnp.linalg.inv(A) =", np.linalg.inv(A), sep="\n")
print("\ndeterminant =", np.linalg.det(A).round(4))

# Solving A w = b.  Two ways:
w_inv   = np.linalg.inv(A) @ b      # mathematically fine, numerically worse
w_solve = np.linalg.solve(A, b)     # preferred: faster and more stable
print("\nvia inverse :", w_inv.round(6))
print("via solve   :", w_solve.round(6))
print("check A@w = b:", np.allclose(A @ w_solve, b))

A.T (transpose) =
[[2. 1.]
 [1. 3.]]

A @ A  (matrix product) =
[[ 5.  5.]
 [ 5. 10.]]

A * A  (ELEMENT-wise!)  =
[[4. 1.]
 [1. 9.]]

np.linalg.inv(A) =
[[ 0.6 -0.2]
 [-0.2  0.4]]

determinant = 5.0

via inverse : [1. 3.]
via solve   : [1. 3.]
check A@w = b: True


> **Practical note.** `@` is matrix multiplication; `*` is element-wise. Confusing the two is
> a silent bug — the shapes often still work. And prefer `np.linalg.solve(A, b)` over
> `inv(A) @ b`: forming an explicit inverse is slower and loses precision. You will see why
> this matters in Lab 2 when $\mathbf{X}^{\top}\mathbf{X}$ is close to singular.

---
### Exercises — Set A (NumPy)

Fill in each cell where you see `# TODO`, then run it. The `assert` statements tell you
whether your answer is correct — if a cell runs without error, you have it right.

In [11]:
# A1. Create a 1-D array of the 20 even numbers from 2 to 40 inclusive.
#     Then report its sum, mean, min and max.

evens = None   # TODO

assert evens.shape == (20,), "expected 20 elements"
assert evens[0] == 2 and evens[-1] == 40
print("evens =", evens)
print(f"sum={evens.sum()}  mean={evens.mean()}  min={evens.min()}  max={evens.max()}")

AttributeError: 'NoneType' object has no attribute 'shape'

In [12]:
# A2. Build a 5x5 matrix whose entry (i, j) equals i * 5 + j  (so it counts 0..24 row-wise).
#     Then extract (a) the third column, (b) the 3x3 block in the bottom-right corner,
#     (c) every element greater than 15.

M5    = None   # TODO
col3  = None   # TODO  -- the column at index 2
block = None   # TODO  -- rows 2..4, columns 2..4
big15 = None   # TODO  -- boolean mask

assert M5.shape == (5, 5) and M5[3, 2] == 17
assert col3.tolist() == [2, 7, 12, 17, 22]
assert block.shape == (3, 3) and block[0, 0] == 12
assert big15.tolist() == list(range(16, 25))
print(M5, "\n\ncol3 =", col3, "\n\nblock =", block, "\n\n>15 =", big15, sep="")

AttributeError: 'NoneType' object has no attribute 'shape'

In [13]:
# A3. VECTORISE THIS. The loop below computes the Euclidean distance from every row of
#     `points` to `centre`. Rewrite it as a single NumPy expression -- no Python loop.
#     Hint: broadcasting, then np.sum(..., axis=1) and np.sqrt.
#     (This is exactly the inner step of K-Nearest Neighbours, which you meet in Lab 4.)

points = rng.normal(size=(1000, 3))
centre = np.array([0.5, -0.2, 1.0])

# --- reference implementation (do not modify) ---
slow = np.empty(len(points))
for i in range(len(points)):
    diff = points[i] - centre
    slow[i] = np.sqrt((diff ** 2).sum())

fast = None   # TODO: one vectorised expression

assert fast.shape == (1000,)
assert np.allclose(slow, fast), "your vectorised version disagrees with the loop"
print("match! first five distances:", fast[:5].round(4))

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
# A4. Standardise the columns of `raw` to zero mean and unit standard deviation,
#     using broadcasting (no loops, no sklearn).

raw = rng.normal(loc=[5, 100, -3], scale=[1, 25, 0.5], size=(200, 3))

standardised = None   # TODO

assert standardised.shape == raw.shape
assert np.allclose(standardised.mean(axis=0), 0, atol=1e-12)
assert np.allclose(standardised.std(axis=0),  1, atol=1e-12)
print("means:", standardised.mean(axis=0).round(12))
print("stds :", standardised.std(axis=0).round(12))

In [ ]:
# A5. Solve the linear system by hand-rolling the normal equation.
#     Given the design matrix X_ls and targets y_ls below, compute
#         w = (X^T X)^{-1} X^T y
#     using np.linalg.solve (NOT np.linalg.inv), and confirm it matches lstsq.
#     This is a preview of Lab 2.

X_ls = np.column_stack([np.ones(50), np.linspace(0, 10, 50)])   # [1, x] -> intercept + slope
true_w = np.array([5.0, 3.0])                                    # y = 5 + 3x + noise
y_ls = X_ls @ true_w + rng.normal(scale=0.5, size=50)

w_hat = None   # TODO

w_ref, *_ = np.linalg.lstsq(X_ls, y_ls, rcond=None)
assert np.allclose(w_hat, w_ref), "does not match numpy's least-squares solution"
print("true      w =", true_w)
print("estimated w =", w_hat.round(4))

**Question A6.** In your own words: why did the vectorised version in A3 beat the loop, and
what does that imply for how you will write the training loops in later labs?

*Your answer:*

<!-- write 2-3 sentences here -->


---
## 3. Pandas — working with real tabular data

NumPy arrays are homogeneous and unlabelled. Real datasets have **named columns of mixed
types**, missing values, and categories. That is what Pandas is for.

* **`Series`** — a labelled 1-D column.
* **`DataFrame`** — a table of Series sharing one index.

### 3.1 The Iris dataset

We use Fisher's Iris data (1936): 150 flowers, 4 measurements each, 3 species. It is small
enough to read on one screen and is the standard first classification dataset — you will meet
it again in Labs 3–5.

In [14]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.copy()

# The target arrives as 0/1/2; attach readable species names as well.
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print("shape:", df.shape)
df.head()

shape: (150, 6)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [15]:
print("--- dtypes ---")
print(df.dtypes)
print("\n--- info ---")
df.info()

--- dtypes ---
sepal length (cm)     float64
sepal width (cm)      float64
petal length (cm)     float64
petal width (cm)      float64
target                  int64
species              category
dtype: object

--- info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   sepal length (cm)  150 non-null    float64 
 1   sepal width (cm)   150 non-null    float64 
 2   petal length (cm)  150 non-null    float64 
 3   petal width (cm)   150 non-null    float64 
 4   target             150 non-null    int64   
 5   species            150 non-null    category
dtypes: category(1), float64(4), int64(1)
memory usage: 6.3 KB


In [16]:
# describe() is your first look at every numeric column at once
df.describe().T

,count,mean,std,min,25%,50%,75%,max
sepal length (cm),150.0,5.843,0.828,4.3,5.1,5.80,6.4,7.9
sepal width (cm),150.0,3.057,0.436,2.0,2.8,3.00,3.3,4.4
petal length (cm),150.0,3.758,1.765,1.0,1.6,4.35,5.1,6.9
petal width (cm),150.0,1.199,0.762,0.1,0.3,1.30,1.8,2.5
target,150.0,1.000,0.819,0.0,0.0,1.00,2.0,2.0


> **Read `describe()` like a checklist.** Compare `mean` with `50%` (a big gap suggests skew);
> look at `min`/`max` for impossible values (a negative petal length would be a data error);
> and compare `std` across columns — very different scales are a warning that distance-based
> models (KNN, SVM, K-Means) will need standardisation.

### 3.2 Selecting data

Three ways, and the difference matters:

| Syntax | Selects by | Example |
|---|---|---|
| `df["col"]` | column name | `df["sepal length (cm)"]` |
| `df.loc[rows, cols]` | **label** | `df.loc[0:5, "species"]` |
| `df.iloc[rows, cols]` | **integer position** | `df.iloc[0:5, -1]` |

Note the trap: `loc` slices are **inclusive** of the end label, `iloc` slices are exclusive
(like normal Python).

In [17]:
# Tidy the column names once -- spaces and brackets make everything harder to type
df.columns = [c.replace(" (cm)", "").replace(" ", "_") for c in df.columns]
print(list(df.columns))

print("\nsingle column (a Series):")
print(df["sepal_length"].head(3))

print("\ntwo columns (a DataFrame):")
print(df[["sepal_length", "species"]].head(3))

print("\n.loc  rows 0-2 by label, two named columns:")
print(df.loc[0:2, ["petal_length", "species"]])

print("\n.iloc rows 0-2 by position, first two columns:")
print(df.iloc[0:3, 0:2])

['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'target', 'species']

single column (a Series):
0    5.1
1    4.9
2    4.7
Name: sepal_length, dtype: float64

two columns (a DataFrame):
   sepal_length species
0           5.1  setosa
1           4.9  setosa
2           4.7  setosa

.loc  rows 0-2 by label, two named columns:
   petal_length species
0           1.4  setosa
1           1.4  setosa
2           1.3  setosa

.iloc rows 0-2 by position, first two columns:
   sepal_length  sepal_width
0           5.1          3.5
1           4.9          3.0
2           4.7          3.2


In [18]:
# Boolean filtering -- the Pandas equivalent of a SQL WHERE clause
big_petals = df[df["petal_length"] > 5.0]
print("flowers with petal_length > 5.0 :", len(big_petals))
print(big_petals["species"].value_counts())

# Multiple conditions: & | ~ with brackets around each condition
selected = df[(df["species"] == "virginica") & (df["sepal_width"] < 3.0)]
print("\nvirginica with narrow sepals :", len(selected))
selected.head()

flowers with petal_length > 5.0 : 42
species
virginica     41
versicolor     1
setosa         0
Name: count, dtype: int64

virginica with narrow sepals : 21


,sepal_length,sepal_width,petal_length,petal_width,target,species
101,5.8,2.7,5.1,1.9,2,virginica
103,6.3,2.9,5.6,1.8,2,virginica
106,4.9,2.5,4.5,1.7,2,virginica
107,7.3,2.9,6.3,1.8,2,virginica
108,6.7,2.5,5.8,1.8,2,virginica


### 3.3 Missing values

Real data has holes. Pandas represents them as `NaN`. There is no universally right fix —
the choice is a modelling decision you must be able to justify.

In [19]:
# Deliberately punch holes in a copy so we can practise
dirty = df.copy()
holes = rng.choice(len(dirty), size=12, replace=False)
dirty.loc[holes[:6],  "sepal_width"]  = np.nan
dirty.loc[holes[6:], "petal_width"]   = np.nan

print("missing values per column:")
print(dirty.isna().sum())
print("\nrows with any missing value:", dirty.isna().any(axis=1).sum())

missing values per column:
sepal_length    0
sepal_width     6
petal_length    0
petal_width     6
target          0
species         0
dtype: int64

rows with any missing value: 12


In [20]:
# Option 1: drop rows that have any missing value -- simple, but throws away data
dropped = dirty.dropna()
print("dropna  :", dirty.shape, "->", dropped.shape, f"(lost {len(dirty)-len(dropped)} rows)")

# Option 2: impute with the column mean -- keeps every row, but shrinks the variance
filled = dirty.fillna(dirty[["sepal_width", "petal_width"]].mean())
print("fillna  : missing after imputation =", int(filled.isna().sum().sum()))

# Option 3 (usually better): impute per group, here per species
by_species = dirty.copy()
for col in ["sepal_width", "petal_width"]:
    by_species[col] = by_species.groupby("species", observed=True)[col].transform(
        lambda s: s.fillna(s.mean())
    )
print("grouped : missing after imputation =", int(by_species.isna().sum().sum()))

print("\nsepal_width mean -- original / global fill / grouped fill:")
print(f"  {df['sepal_width'].mean():.4f}  {filled['sepal_width'].mean():.4f}  {by_species['sepal_width'].mean():.4f}")

dropna  : (150, 6) -> (138, 6) (lost 12 rows)
fillna  : missing after imputation = 0
grouped : missing after imputation = 0

sepal_width mean -- original / global fill / grouped fill:
  3.0573  3.0618  3.0572


> **A warning you will need in Week 12.** Imputing with a mean computed over the *whole*
> dataset — before you split into train and test — leaks information from the test set into
> training. The correct order is: split first, then fit the imputer on the training data only.
> We will formalise this as **data leakage** in Theory Week 12.

### 3.4 Grouping and aggregation

`groupby` implements *split → apply → combine*, the same idea as SQL's `GROUP BY`.

In [21]:
# One statistic per group
print(df.groupby("species", observed=True)["petal_length"].mean(), "\n")

# Several statistics, several columns
summary = df.groupby("species", observed=True).agg(
    n              = ("sepal_length", "size"),
    sepal_len_mean = ("sepal_length", "mean"),
    sepal_len_std  = ("sepal_length", "std"),
    petal_len_mean = ("petal_length", "mean"),
    petal_len_max  = ("petal_length", "max"),
).round(3)
summary

species
setosa        1.462
versicolor    4.260
virginica     5.552
Name: petal_length, dtype: float64 



,n,sepal_len_mean,sepal_len_std,petal_len_mean,petal_len_max
species,,,,,
setosa,50,5.006,0.352,1.462,1.9
versicolor,50,5.936,0.516,4.260,5.1
virginica,50,6.588,0.636,5.552,6.9


> Look at `petal_len_mean` across the three species: 1.46, 4.26, 5.55. Those groups barely
> overlap. That single column already separates the species almost perfectly — which is why
> Iris is an *easy* classification problem, and why you should always look at group statistics
> before you reach for a complicated model.

### 3.5 Merging and derived columns

In [22]:
# A small lookup table to join on
info = pd.DataFrame({
    "species": ["setosa", "versicolor", "virginica"],
    "region":  ["Arctic", "Temperate", "Temperate"],
    "code":    ["SET", "VER", "VIR"],
})

merged = df.merge(info, on="species", how="left")
print("shape after merge:", merged.shape)
print(merged[["sepal_length", "species", "region", "code"]].head(3))

# Derived (engineered) features -- Week 12 will call this feature engineering
merged["sepal_ratio"] = merged["sepal_length"] / merged["sepal_width"]
merged["petal_area"]  = merged["petal_length"] * merged["petal_width"]
merged["size_class"]  = pd.cut(merged["petal_area"],
                               bins=[0, 2, 8, np.inf],
                               labels=["small", "medium", "large"])

print("\n", merged.groupby("size_class", observed=True)["species"].value_counts().unstack(fill_value=0), sep="")

shape after merge: (150, 8)
   sepal_length species  region code
0           5.1  setosa  Arctic  SET
1           4.9  setosa  Arctic  SET
2           4.7  setosa  Arctic  SET

species     setosa  versicolor  virginica
size_class                               
small           50           0          0
medium           0          47          4
large            0           3         46


---
### Exercises — Set B (Pandas)

In [ ]:
# B1. How many flowers of each species have a sepal_length above the overall mean
#     sepal_length? Return a Series indexed by species.

above_mean_counts = None   # TODO

assert isinstance(above_mean_counts, pd.Series)
assert above_mean_counts.sum() == (df["sepal_length"] > df["sepal_length"].mean()).sum()
print(above_mean_counts)

In [ ]:
# B2. Build a table with one row per species and these columns:
#       n, petal_length_mean, petal_width_mean, petal_length_std
#     Round to 3 decimals.

petal_summary = None   # TODO

assert list(petal_summary.index) == ["setosa", "versicolor", "virginica"]
assert set(petal_summary.columns) == {"n", "petal_length_mean", "petal_width_mean", "petal_length_std"}
petal_summary

In [ ]:
# B3. Add a column `petal_length_z` holding the z-score of petal_length computed
#     WITHIN each species (subtract that species' mean, divide by that species' std).
#     Hint: groupby(...).transform(...)

df_ex = df.copy()
df_ex["petal_length_z"] = None   # TODO

check = df_ex.groupby("species", observed=True)["petal_length_z"].mean()
assert np.allclose(check.values, 0, atol=1e-10), "within-species means should be 0"
print(df_ex.groupby("species", observed=True)["petal_length_z"].agg(["mean", "std"]).round(6))

In [ ]:
# B4. Which single feature separates the three species best?
#     For each of the four numeric features compute the ratio
#           (spread of the species means) / (average within-species spread)
#     i.e. std of the three group means, divided by the mean of the three group stds.
#     Higher = better separation. Report a Series sorted from best to worst.
#     (This is a hand-rolled version of the F-statistic used by feature-selection tools.)

features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
separation = None   # TODO -- a pd.Series indexed by feature name, sorted descending

assert isinstance(separation, pd.Series) and len(separation) == 4
print(separation.round(3))
print("\nBest separating feature:", separation.index[0])

---
## 4. Exploratory data analysis

EDA is where you decide *what problem you actually have* before choosing a model. Four
questions worth asking of any new dataset:

1. **Distribution** — is each feature symmetric, skewed, bimodal? (histogram, KDE)
2. **Outliers** — are there impossible or extreme values? (box plot)
3. **Relationships** — which features move together? (scatter, correlation)
4. **Separability** — do the classes actually differ in these features? (colour by class)

### 4.1 Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(data=df, x=col, hue="species", bins=20,
                 multiple="layer", alpha=.55, ax=ax, legend=(ax is axes[0, 0]))
    ax.set_title(col)
fig.suptitle("Feature distributions by species", fontweight="bold")
fig.tight_layout()
plt.show()

**Read the plots.** `petal_length` and `petal_width` show setosa sitting completely apart
from the other two, while `sepal_width` overlaps almost totally. A classifier using only
`sepal_width` would struggle; one using the petal features would find setosa trivial.

### 4.2 Outliers and spread

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=df[features], ax=axes[0])
axes[0].set_title("All features on one axis — note the different scales")
axes[0].set_ylabel("cm")

sns.boxplot(data=df, x="species", y="sepal_width", ax=axes[1])
axes[1].set_title("sepal_width by species — the dots are outliers")

fig.tight_layout()
plt.show()

In [ ]:
# The 1.5 x IQR rule that the box plot draws -- computed explicitly
q1, q3 = df["sepal_width"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = df[(df["sepal_width"] < low) | (df["sepal_width"] > high)]
print(f"Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}   fences: [{low:.2f}, {high:.2f}]")
print(f"{len(outliers)} outlier(s) flagged:")
print(outliers[["sepal_width", "species"]])

> **Do not delete outliers reflexively.** A flagged point may be a measurement error (drop or
> fix it) or a genuine rare case (keep it — it may be exactly what you are trying to detect,
> as in the fraud-detection example from Theory Week 4).

### 4.3 Relationships and correlation

In [ ]:
sns.pairplot(df, hue="species", vars=features, corner=True,
             plot_kws=dict(s=22, alpha=.75), height=1.9)
plt.suptitle("Pairwise relationships", y=1.01, fontweight="bold")
plt.show()

In [ ]:
corr = df[features].corr(numeric_only=True)

plt.figure(figsize=(5.5, 4.2))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=.5)
plt.title("Pearson correlation between features", fontweight="bold")
plt.tight_layout()
plt.show()

print("Strongest pair:")
pairs = corr.where(~np.eye(len(corr), dtype=bool)).stack().sort_values(ascending=False)
print(pairs.head(1))

> **Correlation is not causation, and $r$ only measures a *linear* relationship.** Two features
> can have $r \approx 0$ and still be perfectly related (e.g. $y = x^2$ on a symmetric range).
> Also note `petal_length` and `petal_width` correlate at ~0.96: they carry nearly the same
> information. That redundancy is precisely what **PCA** (Theory Week 9, Lab 8) exploits.

---
### Exercises — Set C (EDA)

In [ ]:
# C1. Produce a scatter plot of petal_length (x) against petal_width (y), coloured by
#     species, with axis labels, a title and a legend.

# TODO: your plot here


In [ ]:
# C2. Compute the correlation between sepal_length and petal_length
#     (a) over the whole dataset, and (b) within each species separately.
#     Print both.

overall_corr = None   # TODO
within_corr  = None   # TODO -- a Series indexed by species

assert isinstance(within_corr, pd.Series) and len(within_corr) == 3
print(f"overall correlation : {overall_corr:.3f}")
print("\nwithin species:")
print(within_corr.round(3))

**Question C3.** In C2 the overall correlation is much higher than the correlation inside any
single species. Explain why. (This effect has a name — look up *Simpson's paradox* — and it is
a standing warning about aggregating across groups.)

*Your answer:*

<!-- write 2-3 sentences here -->


---
## 5. Mini-EDA on a regression dataset

Everything so far used a *classification* dataset. Now repeat the workflow on a **regression**
dataset, where the target is continuous — the setting for Lab 2.

> **A note on the Boston Housing dataset.** Older tutorials (and an earlier draft of this lab
> plan) use `load_boston`. That dataset was **removed from scikit-learn in version 1.2**
> because one of its features encodes race in a way that is ethically indefensible and invites
> misuse. We use the **diabetes** dataset instead, which ships with scikit-learn and needs no
> download. If you want a housing dataset, `fetch_california_housing()` is the sanctioned
> replacement — but it downloads on first use, so it needs internet access.

In [ ]:
from sklearn.datasets import load_diabetes

dia = load_diabetes(as_frame=True)
ddf = dia.frame.copy()          # 10 baseline features + column `target`

print(dia.DESCR[:900])
print("\nshape:", ddf.shape)
ddf.head()

In [ ]:
# The features here are already mean-centred and scaled -- check that claim yourself
print(ddf.drop(columns="target").describe().T[["mean", "std", "min", "max"]].round(4))
print("\ntarget (disease progression one year on):")
print(ddf["target"].describe().round(2))

### Your task

Answer the five questions below **in code and in words**. This is the part of the notebook that
carries most of the marks, because it asks you to interpret rather than to recall.

In [ ]:
# M1. Plot the distribution of `target`. Is it symmetric or skewed?
#     Report its mean, median and skew.

# TODO


In [ ]:
# M2. Which three features correlate most strongly with `target` (by absolute correlation)?

top3 = None   # TODO -- a Series of the three strongest, with their signed correlations

assert len(top3) == 3
print(top3.round(3))

In [ ]:
# M3. Draw a scatter plot of the single most correlated feature against `target`,
#     with a fitted straight line on top (sns.regplot does both).
#     Does a linear model look plausible here?

# TODO


In [ ]:
# M4. Are any two FEATURES strongly correlated with each other (|r| > 0.5)?
#     List those pairs. Why does this matter for a linear model?
#     (Theory Week 3: correlated predictors make coefficients unstable -- multicollinearity.)

strong_pairs = None   # TODO -- a Series or DataFrame of the qualifying pairs

print(strong_pairs)

**M5. Write-up (required).** In 150–200 words, summarise what you learned about the diabetes
dataset from this EDA. Cover at least: the shape of the target distribution; which features look
most predictive and how strongly; any redundancy between features; and one concrete thing you
would check or do before fitting a model in Lab 2.

*Your answer:*

<!-- write your 150-200 word summary here -->


---
## 6. Submission checklist

Before you submit, confirm:

- [ ] The notebook is renamed `Lab01_<your-roll-number>.ipynb`.
- [ ] `Kernel → Restart & Run All` completes with **no errors** (do this last — it proves your
      notebook runs top to bottom, which is how it will be marked).
- [ ] All `# TODO` markers are gone and every `assert` passes.
- [ ] Questions **A6**, **C3** and **M5** are answered in the markdown cells provided.
- [ ] Every plot has axis labels and a title.
- [ ] Your name and roll number are in the cell below.

### Marking rubric (2% of the course total)

| Criterion | Weight |
|---|---|
| Exercise Sets A & B correct (NumPy + Pandas) | 40% |
| Set C and plots — correct, labelled, readable | 20% |
| Mini-EDA M1–M4 correct | 20% |
| Written answers A6, C3, M5 — reasoning, not just description | 20% |

### Before next week

* **Read:** Theory Week 3 slides (Linear Regression) — `theory/lectures/W03_Part1.pdf`.
* **Revise:** the normal equation $\mathbf{w} = (\mathbf{X}^{\top}\mathbf{X})^{-1}\mathbf{X}^{\top}\mathbf{y}$
  and Exercise A5 — Lab 2 starts exactly there.
* **Optional practice:** repeat Section 4 on `load_wine()` or `load_breast_cancer()`.

### Further reading

* NumPy — *the absolute basics for beginners*: <https://numpy.org/doc/stable/user/absolute_beginners.html>
* Pandas — *10 minutes to pandas*: <https://pandas.pydata.org/docs/user_guide/10min.html>
* seaborn tutorial: <https://seaborn.pydata.org/tutorial.html>
* Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*, Ch. 2–3: <https://mml-book.github.io/>


In [ ]:
# Fill this in before submitting.
STUDENT_NAME = "..."
ROLL_NUMBER  = "..."
SESSION      = "..."

print(f"Submitted by {STUDENT_NAME} ({ROLL_NUMBER}), session {SESSION}")
print("Lab 1 — Python for Machine Learning — CSE 814, University of Chittagong")